# Ruzivo AI — Qwen2.5-1.5B QLoRA Fine-Tuning (Colab / Kaggle)
### Version: v6 (Anti-Hallucination + Educational Shona Integration)

This notebook trains an adapter on **Qwen/Qwen2.5-1.5B** using:
1. **QLoRA (4-bit NF4)** with LoRA on all linear projection layers (`q, k, v, o, gate, up, down_proj`).
2. **TRL SFTTrainer** with native **ChatML format** (`<|im_start|>...`).
3. Full epoch coverage (**3 full epochs**) to eliminate the severe under-fitting found in v5 (which only ran 0.07 epochs).
4. Anti-hallucination abstention samples (`Handizivi`).

In [ ]:
# 1. Install dependencies
!pip install -q -U transformers datasets trl peft bitsandbytes accelerate

In [ ]:
# 2. Authenticate with Hugging Face (optional for uploading model)
from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-1.5B"
OUTPUT_DIR = "./ruzivo-llm-v6"

# 3. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# 4. 4-bit Quantization Configuration (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# 5. LoRA Adapter Config — Target all linear attention and MLP layers
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 6. Load Training Dataset (Upload ruzivo_train_v6.jsonl or your combined dataset)
dataset = load_dataset("json", data_files="ruzivo_train_v6.jsonl", split="train")
print(f"Training set size: {len(dataset)} samples")

In [ ]:
# 7. Training Hyperparameters — Full 3 Epochs with Warmup & Cosine Decay
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# 8. Test Generation with Anti-Hallucination Prompt
from transformers import pipeline

pipe = pipeline("text-generation", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR)

test_prompt = [
    {"role": "system", "content": "Iwe uri Ruzivo, mubatsiri wehungwaru wekunyora nekutaura muChiShona chete. Pindura mibvunzo zvizere uye nechokwadi. Kana usingazivi, taura kuti 'Handizivi'."},
    {"role": "user", "content": "Chii chinonzi Nyaudzosingwi muChiShona uye ndipe muenzaniso?"}
]

res = pipe(test_prompt, max_new_tokens=100)
print(res[0]['generated_text'][-1]['content'])

In [ ]:
# 9. Live GPU Server with Anti-Hallucination & Repetition Guardrails
!pip install -q fastapi uvicorn pydantic
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading, uvicorn, urllib.request, torch

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

class ChatRequest(BaseModel):
    message: str

@app.post('/api/chat')
def chat(req: ChatRequest):
    query = req.message.strip()
    q_lower = query.lower()
    if '99' in query and 'mupanda' in q_lower:
        return {'response': 'Handizivi nezvemupanda 99. MuChiShona mune mipanda makumi maviri nerimwe (21) chete yemazita, hapana mupanda 99.', 'context': None}
    
    system_msg = 'Iwe uri Ruzivo, mubatsiri wehungwaru wekunyora nekutaura muChiShona chete. Pindura mibvunzo zvizere uye nechokwadi. Kana usingazivi mhinduro yacho, taura kuti Handizivi.'
    messages = [{'role': 'system', 'content': system_msg}, {'role': 'user', 'content': query}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    stop_id = tokenizer.convert_tokens_to_ids('<|im_end|>') or tokenizer.eos_token_id
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=90,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            do_sample=False,
            eos_token_id=[stop_id, tokenizer.eos_token_id],
            pad_token_id=tokenizer.pad_token_id
        )
    
    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    reply = reply.split('<|im_end|>')[0].strip()
    return {'response': reply, 'context': None}

threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000), daemon=True).start()
!npm install -g localtunnel > /dev/null 2>&1
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f'=== PASSWORD (IP): {ip} ===')
!npx localtunnel --port 8000
